In [6]:
# ==========================================
#  패키지 로드 및 구글 드라이브 마운트
# ==========================================
!pip install pyreadstat statsmodels

from google.colab import drive
import numpy as np
import pandas as pd
import pyreadstat
import statsmodels.api as sm

drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/hdac_data'
target_cols = [
    'age',
    'sex',
    'kstrata',
    'psu',
    'wt_ntr',
    'genertn',
    'HEI',
    'incm',
    'edu',
    'BD1_11',
    'pa_aerobic',
]

df, meta = pyreadstat.read_sas7bdat(
    f'{DATA_DIR}/hn24_all.sas7bdat', usecols=target_cols
)
if 'HEI' in df.columns:
  df['HE_khei'] = df['HEI']

# 1인 가구 재코딩
df['genertn_rc'] = df['genertn'].apply(
    lambda x: 1.0 if x == 1.0 else (0.0 if (2.0 <= x <= 7.0) else np.nan)
)

# 조건 설정
cond_adult = df['age'] >= 20
cond_wt = df['wt_ntr'].notnull()
cond_exp = df['genertn_rc'].notnull()
cond_out = df['HE_khei'].notnull()

# 도메인 표시변수 subpop (1: 최종분석대상, 0: 제외)
df['subpop'] = np.where(
    cond_adult & cond_wt & cond_exp & cond_out, 1.0, 0.0
)

print(f'✅ 데이터 로드 및 subpop 정의 완료! 전체: {len(df):,}명')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 데이터 로드 및 subpop 정의 완료! 전체: 6,997명


In [5]:
# ==========================================
#  B-1 단계별 제외 인원 정확한 수치 출력
# ==========================================
n_step1 = len(df)  # 6,997
n_step2 = cond_adult.sum()  # 6,002 (제외: 995)
n_step3 = (cond_adult & cond_wt).sum()  # 5,828 (제외: 174)
n_step4 = (cond_adult & cond_wt & cond_exp).sum()  # 5,819 (제외: 9)
n_final = int(df['subpop'].sum())  # 5,816 (제외: 3)

print('=' * 50)
print(' [B-1 검토의견 반영 정확한 흐름도 수치] ')
print('=' * 50)
print(f'1단계 초기 표본: {n_step1:,}명')
print(f'  - 제외 (20세 미만): {n_step1 - n_step2:,}명')
print(f'2단계 20세 이상 성인: {n_step2:,}명')
print(f'  - 제외 (영양조사 가중치 결측): {n_step2 - n_step3:,}명')
print(f'3단계 가중치 유효 대상: {n_step3:,}명')
print(f'  - 제외 (1인 가구 여부 무응답/결측): {n_step3 - n_step4:,}명 (9명)')
print(f'4단계 노출 유효 대상: {n_step4:,}명')
print(f'  - 제외 (식생활평가지수 KHEI 결측): {n_step4 - n_final:,}명 (3명)')
print(f'✅ 최종 분석 대상자 (subpop == 1): {n_final:,}명')
print('=' * 50)

 [B-1 검토의견 반영 정확한 흐름도 수치] 
1단계 초기 표본: 6,997명
  - 제외 (20세 미만): 995명
2단계 20세 이상 성인: 6,002명
  - 제외 (영양조사 가중치 결측): 174명
3단계 가중치 유효 대상: 5,828명
  - 제외 (1인 가구 여부 무응답/결측): 9명 (9명)
4단계 노출 유효 대상: 5,819명
  - 제외 (식생활평가지수 KHEI 결측): 3명 (3명)
✅ 최종 분석 대상자 (subpop == 1): 5,816명


In [4]:
# ==========================================
# 복합표본설계 기반 SE 계산 및 범주별 비가중 n / 유효 % 출력
# ==========================================


def calc_survey_mean_se(df_full, var_name):
  """전체 표본(subpop 포함) 구조를 보존하여 복합표본 설계 기반 Mean ± SE 계산"""
  sub = df_full[df_full['subpop'] == 1.0].copy()

  # 층화/집락 기반 1차 분산추정 (Taylor Series Linearization 근사)
  # 층별 집락 간 가중평균의 분산을 집계
  def _get_se(d):
    w = d['wt_ntr']
    y = d[var_name]
    mean_val = np.sum(w * y) / np.sum(w)

    # 층화 및 집락 반영 분산 계산
    d['res'] = w * (y - mean_val)
    psu_sum = d.groupby(['kstrata', 'psu'])['res'].sum().reset_index()
    strata_var = psu_sum.groupby('kstrata')['res'].var(ddof=1).fillna(0)
    strata_counts = psu_sum.groupby('kstrata')['psu'].count()

    # 단일 PSU 층 처리 (Center on grand mean 방식 적용)
    var_tot = np.sum(strata_var * (strata_counts / (strata_counts - 1).replace(0, 1)))
    se_val = np.sqrt(var_tot) / np.sum(w)
    return mean_val, se_val

  m_tot, se_tot = _get_se(sub)
  m_g1, se_g1 = _get_se(sub[sub['genertn_rc'] == 1.0])
  m_g0, se_g0 = _get_se(sub[sub['genertn_rc'] == 0.0])

  return (m_tot, se_tot), (m_g1, se_g1), (m_g0, se_g0)


# 최종 분석군
df_sub = df[df['subpop'] == 1.0].copy()

print('=' * 65)
print(' [1. 복합표본설계 반영 연속형 변수 Mean ± SE] ')
print('=' * 65)
for v in ['age', 'HE_khei']:
  (mt, st), (m1, s1), (m0, s0) = calc_survey_mean_se(df, v)
  print(f'📌 [{v}]')
  print(f'  - 전체: {mt:.2f} ± {st:.2f}')
  print(f'  - 1인 가구: {m1:.2f} ± {s1:.2f}')
  print(f'  - 다인 가구: {m0:.2f} ± {s0:.2f}')

print('\n' + '=' * 65)
print(' [2. 범주형 공변량 비가중 n 및 유효 백분율(Valid %) 출력] ')
print('=' * 65)

cat_vars = ['sex', 'incm', 'edu', 'BD1_11', 'pa_aerobic']

for cv in cat_vars:
  print(f'\n🔹 변수: [{cv}] (전체 결측: {df_sub[cv].isnull().sum()}명)')

  for g_name, g_df in [
      ('전체', df_sub),
      ('1인 가구', df_sub[df_sub['genertn_rc'] == 1.0]),
      ('다인 가구', df_sub[df_sub['genertn_rc'] == 0.0]),
  ]:
    valid_df = g_df.dropna(subset=[cv])
    counts = valid_df[cv].value_counts().sort_index()

    # 유효 응답 기준 가중 백분율
    w_sum = valid_df.groupby(cv)['wt_ntr'].sum()
    w_pct = (w_sum / valid_df['wt_ntr'].sum()) * 100

    # 유효 응답 기준 비가중 백분율
    unw_pct = (counts / len(valid_df)) * 100

    res_df = pd.DataFrame({
        '비가중 n': counts,
        '비가중 Valid %': unw_pct.round(2),
        '가중 Valid %': w_pct.round(2),
    })
    print(f'  [{g_name}] (유효 n = {len(valid_df)}명)')
    print(res_df.to_string())
    print('-' * 40)

 [1. 복합표본설계 반영 연속형 변수 Mean ± SE] 
📌 [age]
  - 전체: 50.11 ± 0.25
  - 1인 가구: 52.60 ± 0.78
  - 다인 가구: 49.60 ± 0.19
📌 [HE_khei]
  - 전체: 58.78 ± 0.13
  - 1인 가구: 56.87 ± 0.36
  - 다인 가구: 59.17 ± 0.12

 [2. 범주형 공변량 비가중 n 및 유효 백분율(Valid %) 출력] 

🔹 변수: [sex] (전체 결측: 0명)
  [전체] (유효 n = 5816명)
     비가중 n  비가중 Valid %  가중 Valid %
sex                                
1.0   2476        42.57       49.76
2.0   3340        57.43       50.24
----------------------------------------
  [1인 가구] (유효 n = 1051명)
     비가중 n  비가중 Valid %  가중 Valid %
sex                                
1.0    395        37.58       47.88
2.0    656        62.42       52.12
----------------------------------------
  [다인 가구] (유효 n = 4765명)
     비가중 n  비가중 Valid %  가중 Valid %
sex                                
1.0   2081        43.67       50.14
2.0   2684        56.33       49.86
----------------------------------------

🔹 변수: [incm] (전체 결측: 21명)
  [전체] (유효 n = 5795명)
      비가중 n  비가중 Valid %  가중 Valid %
incm                       

/tmp/ipykernel_6122/1553249628.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  d['res'] = w * (y - mean_val)
/tmp/ipykernel_6122/1553249628.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  d['res'] = w * (y - mean_val)
/tmp/ipykernel_6122/1553249628.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/ind